# Generate your own synthetic data with DP-CGANS

**Unlock Your Sensitive Data** · TDCC-SSH workshop · 14 October 2026

Run the cells from top to bottom with **Shift+Enter**. In 15 minutes you will train a generator on a small public dataset, create synthetic rows from it, look at how close they are to the real data, and switch differential privacy on.

## 1. Install the package

In [ ]:
%%capture
!pip install dp-cgans

## 2. Load the data

Census income data on adults: age, education, marital status, sex, type of employer, working hours and income. Public data, so nothing sensitive here. Think of it as a stand-in for a survey you are not allowed to share.

In [ ]:
import pandas as pd

tabular_data = pd.read_csv("https://raw.githubusercontent.com/sunchang0124/syndata-workshop/main/data/census_workshop.csv").sample(2000, random_state=0)
tabular_data.head()

## 3. Train the generator

This is the usage example from the package documentation. Forty rounds of training takes about a minute. Watch the progress bar.

In [ ]:
from dp_cgans import DP_CGAN
import time, os

if os.path.exists("fitted_transformer.pkl"):   # the package reuses an old transformer if it finds one; start fresh
    os.remove("fitted_transformer.pkl")

model = DP_CGAN(
    epochs=30,            # number of training rounds
    batch_size=500,       # rows per training step
    private=False,        # differential privacy off for now
    verbose=False,
    saved_transformer=None,
)

start_time = time.time()
model.fit(tabular_data)
print("Training took", round(time.time() - start_time), "seconds")

model.save("generator.pkl")

## 4. Generate synthetic rows

In [ ]:
syn_data = model.sample(2000)
syn_data.to_csv("syn_data_file.csv", index=None)
syn_data.head()

## 5. Does it look like the real data?

Blue is real, orange is synthetic. After only 30 rounds the generator knows the categories but usually not yet the right proportions.

In [ ]:
import matplotlib.pyplot as plt

def compare(real, synthetic, title):
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, col in zip(axes.ravel(), real.columns):
        if real[col].dtype == object:
            pd.DataFrame({"real": real[col].value_counts(normalize=True),
                          "synthetic": synthetic[col].value_counts(normalize=True)}).plot.bar(ax=ax, legend=False)
        else:
            ax.hist([real[col], synthetic[col]], bins=20, density=True, label=["real", "synthetic"])
        ax.set_title(col)
    axes.ravel()[-1].axis("off")
    axes.ravel()[0].legend(["real", "synthetic"])
    fig.suptitle(title); plt.tight_layout(); plt.show()

compare(tabular_data, syn_data, "After 30 training rounds")

<!-- ## 6. Switch differential privacy on

Same model, `private=True`. The package now adds noise while it learns and prints the privacy guarantee **epsilon** after every round. Smaller epsilon means stronger privacy. Watch it grow as training continues. -->

In [ ]:
# ### Turn on differential privacy parameter ###
# if os.path.exists("fitted_transformer.pkl"):
#     os.remove("fitted_transformer.pkl")

# private_model = DP_CGAN(epochs=30, batch_size=500, private=True, verbose=False, saved_transformer=None)
# private_model.fit(tabular_data)

# private_syn_data = private_model.sample(2000)
# compare(tabular_data, private_syn_data, "After 30 training rounds, with differential privacy")

## 6. A generator trained longer

A generator usually needs many hundreds of rounds. We trained the same model for **1 000 rounds** before the workshop. Load it and compare: this is the quality you can expect from DP-CGANS on a small table.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/sunchang0124/syndata-workshop/main/models/dpcgan_best.pkl", "dpcgan_best.pkl")
trained_model = DP_CGAN.load("dpcgan_best.pkl")

syn_trained = trained_model.sample(len(tabular_data))
compare(tabular_data, syn_trained, "After 1 000 training rounds")

## 7. Would a statistician get the same answers?

Looking similar is not the point. The point is that an analysis done on the synthetic data leads to the same conclusions as on the real data. We run the same analyses on both and put the results side by side. We use 5 000 synthetic rows for stable estimates.

In [ ]:
syn_trained = trained_model.sample(5000)
real = tabular_data

# Descriptive statistics
pd.concat({"real": real.describe().T, "synthetic": syn_trained.describe().T}, axis=1).round(1)

In [ ]:
# Group comparison: are people with a high income older, and do they work more hours?
from scipy.stats import ttest_ind

def group_means(df):
    out = df.groupby("income")[["age", "hours_per_week"]].mean().T
    out["difference"] = out["high"] - out["low"]
    out["p-value"] = [ttest_ind(df.loc[df.income == "high", c], df.loc[df.income == "low", c]).pvalue for c in out.index]
    return out

pd.concat({"real": group_means(real), "synthetic": group_means(syn_trained)}, axis=1).round(3)

In [ ]:
# Cross-tables: share of high income by education and by marital status
def share_high(df, col):
    return pd.crosstab(df[col], df["income"], normalize="index")["high"]

pd.concat({"real": pd.concat([share_high(real, "education"), share_high(real, "marital_status")]),
           "synthetic": pd.concat([share_high(syn_trained, "education"), share_high(syn_trained, "marital_status")])}, axis=1).round(2)

In [ ]:
# Correlations between all variables (categories turned into 0/1 columns): real, synthetic, and the difference
import seaborn as sns

real_corr = pd.get_dummies(real, dtype=float).corr()
syn_corr  = pd.get_dummies(syn_trained, dtype=float).corr().reindex_like(real_corr)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.heatmap(real_corr, ax=axes[0], cmap="RdBu_r", vmin=-1, vmax=1, cbar=False); axes[0].set_title("real")
sns.heatmap(syn_corr,  ax=axes[1], cmap="RdBu_r", vmin=-1, vmax=1, cbar=False, yticklabels=False); axes[1].set_title("synthetic")
sns.heatmap((real_corr - syn_corr).abs(), ax=axes[2], cmap="Reds", vmin=0, vmax=1, yticklabels=False); axes[2].set_title("absolute difference")
plt.tight_layout(); plt.show()

print("average absolute difference in correlation:", round(float((real_corr - syn_corr).abs().mean().mean()), 3), "(0 = identical)")

## Learn more

* `pip install dp-cgans` · https://github.com/sunchang0124/dp_cgans · https://pypi.org/project/dp-cgans/
* Sun C, van Soest J, Dumontier M. *Generating synthetic personal health data using conditional generative adversarial networks combining with differential privacy.* Journal of Biomedical Informatics 143 (2023) 104404.
* metasyn, the other tool in this workshop: https://metasyn.readthedocs.io
* Questions: chang.sun@maastrichtuniversity.nl